# DS6 · Forecasting avanzado: muchas series, modelos globales y jerarquías

**Caso:** TiendaSur vende 20 productos en 5 tiendas = **100 series diarias**. El equipo de compras necesita la demanda de las **próximas 4 semanas** por producto y tienda, con un intervalo para dimensionar el stock de seguridad, y que las cifras de tienda, categoría y total *cuadren entre sí*.

En I6 pronosticaste **una** serie con Holt-Winters y SARIMA. Aquí cambia el problema: con cientos o miles de series no puedes ajustar y vigilar un modelo a mano por serie. Veremos:

1. Simular un panel realista (tendencia, estacionalidad semanal y anual, festivos, promociones).
2. Medir bien: **MASE, WAPE y sesgo**, con *backtesting* de origen rodante.
3. **Modelos locales** (un ETS por serie) frente a **un modelo global con ML** (un único gradient boosting para las 100 series).
4. **Intervalos de predicción:** regresión cuantílica y *conformal*.
5. **Jerarquías y reconciliación** (*bottom-up*, OLS, WLS).
6. Modelos fundacionales de series temporales: dónde encajan y cómo evaluarlos (esqueleto para Colab).

> Se ejecuta en CPU en un par de minutos. Usamos `HistGradientBoostingRegressor` de scikit-learn; con LightGBM el código es casi idéntico (ver apuntes).

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (9, 3.4), "axes.grid": True, "grid.alpha": 0.3})

H = 28                      # horizonte: 4 semanas
FECHAS = pd.date_range("2024-01-01", "2026-09-30", freq="D")
T = len(FECHAS)
print(f"{T} días observados, de {FECHAS[0].date()} a {FECHAS[-1].date()}")

## 1 · El panel: 100 series con estructura jerárquica

Cada serie es un par **producto × tienda**. Las series agregan de forma natural: producto → categoría → total, y tienda → total. Esa *jerarquía* (en realidad «agrupada», porque se cruzan dos ejes) es la que reconciliaremos en la sección 6.

Lo que importa de la simulación: la demanda lleva **efectos que un modelo local no puede ver** (promociones y festivos planificados) y que un modelo global puede recibir como variables.

In [ ]:
TIENDAS = {"Sevilla": "Sur", "Málaga": "Sur", "Madrid": "Centro", "Valencia": "Este", "Bilbao": "Norte"}
PRODUCTOS = {
    "Café": ["Grano 1 kg", "Molido 250 g", "Cápsulas x50", "Descafeinado", "Cold brew"],
    "Té": ["Té verde", "Té negro", "Rooibos", "Matcha", "Infusión frutal"],
    "Menaje": ["Taza", "Cafetera italiana", "Molinillo", "Prensa francesa", "Filtros"],
    "Regalo": ["Pack degustación", "Caja regalo", "Calendario adviento", "Tarjeta regalo", "Set barista"],
}
SERIES = pd.DataFrame(
    [(p, t, c, TIENDAS[t]) for c, ps in PRODUCTOS.items() for p in ps for t in TIENDAS],
    columns=["producto", "tienda", "categoria", "region"],
)
SERIES["producto_c"] = SERIES["producto"].astype("category").cat.codes
SERIES["tienda_c"] = SERIES["tienda"].astype("category").cat.codes
SERIES["cat_c"] = SERIES["categoria"].astype("category").cat.codes
S = len(SERIES)
print(f"{S} series = {SERIES.producto.nunique()} productos x {SERIES.tienda.nunique()} tiendas")
SERIES.head(6)

In [ ]:
def calendario(fechas):
    """Variables de calendario que se conocen de antemano (también para el futuro)."""
    f = pd.DatetimeIndex(fechas)
    fijos = {(1, 1), (1, 6), (5, 1), (8, 15), (10, 12), (11, 1), (12, 6), (12, 8), (12, 25)}
    festivo = np.array([(d.month, d.day) in fijos for d in f])
    cierre = np.array([(d.month, d.day) in {(1, 1), (12, 25)} for d in f])   # tiendas cerradas
    bf = np.zeros(len(f), bool)                       # Black Friday: viernes tras el 4.º jueves de noviembre
    for anio in f.year.unique():
        nov1 = pd.Timestamp(anio, 11, 1)
        jueves4 = nov1 + pd.Timedelta(days=(3 - nov1.weekday()) % 7 + 21)
        bf |= (f == jueves4 + pd.Timedelta(days=1))
    navidad = np.array([(d.month == 12 and 15 <= d.day <= 24) for d in f])
    return pd.DataFrame({
        "dow": f.dayofweek, "mes": f.month, "doy": f.dayofyear, "dom": f.day,
        "festivo": festivo.astype(int), "cierre": cierre.astype(int),
        "bf": bf.astype(int), "navidad": navidad.astype(int),
    })

CAL = calendario(FECHAS)
print("Festivos en el periodo:", int(CAL.festivo.sum()), "| Black Fridays:", int(CAL.bf.sum()))

In [ ]:
def simular(semilla=2026):
    """Ventas diarias (unidades) por producto x tienda: tendencia, estacionalidad semanal y anual,
    festivos, Navidad, Black Friday y promociones, con ruido de conteo sobredisperso."""
    rng = np.random.default_rng(semilla)
    t = np.arange(T)
    nivel = rng.lognormal(np.log(10), 0.8, S)                    # unidades/día de cada serie
    crec = rng.normal(0.06, 0.10, S)                             # crecimiento anual (log)
    tendencia = nivel[:, None] * np.exp(crec[:, None] * t / 365)

    pico = SERIES["categoria"].map({"Café": 15, "Té": 20, "Menaje": 340, "Regalo": 350}).values
    amp = SERIES["categoria"].map({"Café": 0.12, "Té": 0.25, "Menaje": 0.20, "Regalo": 0.35}).values
    anual = 1 + amp[:, None] * np.cos(2 * np.pi * (CAL["doy"].values[None, :] - pico[:, None]) / 365.25)

    base_sem = np.array([0.80, 0.82, 0.90, 0.95, 1.12, 1.40, 1.01])
    sem = base_sem[None, :] * rng.lognormal(0, 0.08, (S, 7))
    sem /= sem.mean(1, keepdims=True)
    semanal = sem[:, CAL["dow"].values]

    promo = np.zeros((S, T))                                     # promociones de 7 días, planificadas
    for i in range(S):
        for inicio in rng.choice(T - 7, size=T // 60, replace=False):
            promo[i, inicio:inicio + 7] = 1
    subida = rng.uniform(1.3, 1.7, S)

    nav = SERIES["categoria"].map({"Café": 0.15, "Té": 0.15, "Menaje": 0.30, "Regalo": 0.90}).values
    fest = (1 - 0.15 * CAL["festivo"].values)[None, :] * (1 + 0.6 * CAL["bf"].values)[None, :]
    fest = fest * (1 + nav[:, None] * CAL["navidad"].values[None, :])
    fest = fest * np.where(CAL["cierre"].values == 1, 0.05, 1.0)[None, :]

    mu = tendencia * anual * semanal * (1 + (subida[:, None] - 1) * promo) * fest
    y = rng.poisson(mu * rng.gamma(15, 1 / 15, mu.shape))        # Poisson mezclado con gamma = binomial negativa
    return y.astype(float), promo

Y, PROMO = simular()
total = pd.Series(Y.sum(0), index=FECHAS)
print(f"Unidades/día: media por serie {Y.mean():.1f}; total TiendaSur {total.mean():.0f}")
print(f"Series con >10 % de días a cero: {int(((Y == 0).mean(1) > 0.10).sum())} de {S}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
total.resample("W").sum().plot(ax=ax[0], title="Total semanal (todas las series)")
pd.Series(Y[0], index=FECHAS)["2025-11-15":"2026-01-31"].plot(ax=ax[1], title=f"Una serie: {SERIES.producto[0]} · {SERIES.tienda[0]}")
plt.tight_layout()
plt.show()

## 2 · Cómo medir (y cómo validar)

- **MASE** (*mean absolute scaled error*): el error medio absoluto dividido por el error medio del **ingenuo estacional dentro de la muestra de entrenamiento**. No depende de la escala (sirve para promediar series pequeñas y grandes) y se lee directamente: `< 1` = mejor que repetir la semana anterior.
- **WAPE** (*weighted absolute percentage error*) = Σ|error| / Σ ventas. Al contrario que el MAPE, no explota con días de venta cero y pondera por volumen: es lo que entiende un director de compras («nos equivocamos un 29 % de las unidades»).
- **Sesgo** = Σ(previsión − real) / Σ real. Un modelo con buen WAPE pero sesgo −10 % genera roturas de stock sistemáticas.

**Backtesting con origen rodante:** en vez de un único corte, usamos 4 orígenes consecutivos separados 28 días. En cada uno el modelo solo ve el pasado y predice las 4 semanas siguientes. Un solo pliegue puede engañar (¿justo cayó un festivo?); cuatro dan una idea de la variabilidad.

In [ ]:
def metricas(real, pred, escala):
    """real, pred: (pliegues, series, H). escala: (pliegues, series) = error medio del ingenuo estacional
    DENTRO de la muestra de entrenamiento (denominador de MASE)."""
    err = pred - real
    return {
        "MASE": float((np.abs(err).mean(2) / escala).mean()),    # <1: mejor que el ingenuo estacional
        "WAPE": float(np.abs(err).sum() / real.sum()),           # error total / ventas totales
        "sesgo": float(err.sum() / real.sum()),                  # >0: sobrestima; <0: subestima
    }

# Pliegues del backtesting con origen rodante: 4 origenes separados H dias, cada uno predice los H siguientes
ORIGENES = [T - 1 - H * k for k in (4, 3, 2, 1)]
REAL = np.stack([Y[:, o + 1:o + 1 + H] for o in ORIGENES])             # (4, S, H)
ESC = np.stack([np.abs(Y[:, o - 364:o + 1][:, 7:] - Y[:, o - 364:o + 1][:, :-7]).mean(1) for o in ORIGENES])
for k, o in enumerate(ORIGENES, 1):
    print(f"pliegue {k}: entrena hasta {FECHAS[o].date()}  ->  evalua {FECHAS[o + 1].date()} .. {FECHAS[o + H].date()}")

## 3 · Modelos locales: referencias y un ETS por serie

Primero las referencias que **cualquier** modelo debe superar. Fíjate en el MASE de las referencias: el del ingenuo estacional no es exactamente 1 porque la escala se mide dentro de la muestra y la evaluación es fuera de ella. Después, un Holt-Winters (ETS) por serie, que es el estándar local: 100 modelos, 100 conjuntos de parámetros, ninguna información compartida.

In [ ]:
def pred_ingenuo(Yh):            # repite el último valor
    return np.repeat(Yh[:, -1:], H, axis=1)

def pred_estacional(Yh):         # repite la última semana
    return np.tile(Yh[:, -7:], (1, H // 7))

def pred_estacional_4s(Yh):      # media de las últimas 4 semanas, por día de la semana
    ult = Yh[:, -28:].reshape(len(Yh), 4, 7).mean(1)
    return np.tile(ult, (1, H // 7))

def pred_ets(Yh):
    """Un Holt-Winters (tendencia amortiguada + estacionalidad semanal) POR serie: modelo local."""
    salida = np.zeros((len(Yh), H))
    for i, serie in enumerate(Yh):
        ajuste = ExponentialSmoothing(serie[-182:] + 0.0, trend="add", damped_trend=True,
                                      seasonal="add", seasonal_periods=7).fit()
        salida[i] = np.clip(ajuste.forecast(H), 0, None)
    return salida

LOCALES = {"ingenuo": pred_ingenuo, "estacional": pred_estacional,
           "estacional_4s": pred_estacional_4s, "ETS local": pred_ets}
PRED = {}
for nombre, f in LOCALES.items():
    t0 = time.time()
    PRED[nombre] = np.stack([f(Y[:, :o + 1]) for o in ORIGENES])
    m = metricas(REAL, PRED[nombre], ESC)
    print(f"{nombre:14s} MASE {m['MASE']:.3f} | WAPE {m['WAPE']:.3f} | sesgo {m['sesgo']:+.3f}   ({time.time() - t0:.1f} s)")

## 4 · Un modelo global con ML

**Idea:** en lugar de 100 modelos, entrenar **uno solo** con las 100 series apiladas. Cada fila es «una serie, desde un origen, para un horizonte `h`». Sus variables:

- **Retardos y ventanas** *del origen* (medias de 7, 28 y 91 días; el mismo día de la semana 1–4 semanas atrás; las mismas fechas hace un año).
- **Calendario y eventos planificados del día objetivo:** día de la semana, mes, festivo, Navidad, Black Friday, cierre, **¿hay promoción?**
- **Identificadores** (producto, tienda, categoría) como variables categóricas: el modelo aprende que el café sube en invierno y el menaje en Navidad.
- **Horizonte `h`**: estrategia *directa* (un modelo para todos los horizontes), que evita acumular error como en la recursiva.

Dos decisiones clave que se olvidan:
1. **Escalar por serie.** El objetivo es `ventas / media de los últimos 28 días`, así una serie de 3 uds/día y otra de 50 aportan al mismo problema. Luego se desescala.
2. **Sin fugas:** al entrenar para un origen `o`, solo entran filas cuyo *objetivo* es ≤ `o`.

In [ ]:
def filas(origenes, con_objetivo=True):
    """Tabla de entrenamiento 'directa': una fila por (serie, origen, horizonte h).
    Todo lo que usa la fila se conoce en el origen o es calendario/promoción planificada."""
    trozos = []
    for o in origenes:
        v7, v28, v91 = (Y[:, o - n + 1:o + 1].mean(1) for n in (7, 28, 91))
        esc = v28 + 0.5                                  # escala de cada serie: así comparables
        cv = Y[:, o - 27:o + 1].std(1) / esc
        promo28 = PROMO[:, o - 27:o + 1].mean(1)
        for h in range(1, H + 1):
            tt = o + h                                   # fecha objetivo
            base = tt - 7 * int(np.ceil(h / 7))          # mismo día de la semana, ya observado
            d = {
                "producto_c": SERIES["producto_c"].values, "tienda_c": SERIES["tienda_c"].values,
                "cat_c": SERIES["cat_c"].values, "log_esc": np.log(esc), "h": np.full(S, h),
                "r7": v7 / esc, "r28": v28 / esc, "r91": v91 / esc, "cv": cv, "promo28": promo28,
                "a364": Y[:, tt - 367:tt - 360].mean(1) / esc,        # mismas fechas hace un año
                "promo_lag": PROMO[:, base], "promo_t": PROMO[:, tt],
                "o": np.full(S, o), "esc": esc,
            }
            for j in range(4):
                d[f"s{j + 1}"] = Y[:, base - 7 * j] / esc            # mismo día, 1..4 semanas atrás
            for c in ("dow", "mes", "doy", "dom", "festivo", "cierre", "bf", "navidad"):
                d[c] = np.full(S, CAL[c].values[tt])
            if con_objetivo:
                d["y"] = Y[:, tt] / esc
            d["serie"] = np.arange(S)
            trozos.append(pd.DataFrame(d))
    return pd.concat(trozos, ignore_index=True)

CATEGORICAS = ["producto_c", "tienda_c", "cat_c"]
VARS = [c for c in filas([400], False).columns if c not in ("o", "esc", "serie")]
t0 = time.time()
ENTRENO = filas(range(400, T - 1 - H + 1, 14))
print(f"{len(ENTRENO):,} filas de entrenamiento, {len(VARS)} variables ({time.time() - t0:.0f} s)")
print("Variables:", VARS)

In [ ]:
def ajustar_global(origen, perdida="poisson", **extra):
    """Entrena con TODAS las series a la vez usando solo objetivos <= origen (sin mirar el futuro)."""
    train = ENTRENO[ENTRENO["o"] + ENTRENO["h"] <= origen]
    parametros = dict(loss=perdida, learning_rate=0.08, max_iter=250, max_leaf_nodes=31,
                      min_samples_leaf=100, l2_regularization=1.0, categorical_features=CATEGORICAS,
                      random_state=0)
    parametros.update(extra)
    return HistGradientBoostingRegressor(**parametros).fit(train[VARS], train["y"]), len(train)

def predecir_global(modelo, origen):
    X = filas([origen], con_objetivo=False)
    ratio = modelo.predict(X[VARS])
    return np.clip(ratio * X["esc"].values, 0, None).reshape(H, S).T      # (series, H)

MODELOS = {}
t0 = time.time()
pred = []
for k, o in enumerate(ORIGENES):
    MODELOS[k], n = ajustar_global(o)
    pred.append(predecir_global(MODELOS[k], o))
    print(f"pliegue {k + 1}: {n:,} filas, {time.time() - t0:.0f} s acumulados")
PRED["global HGB"] = np.stack(pred)

In [ ]:
tabla = pd.DataFrame({n: metricas(REAL, p, ESC) for n, p in PRED.items()}).T.round(3)
tabla["WAPE pliegue 1..4"] = [" ".join(f"{metricas(REAL[k:k+1], p[k:k+1], ESC[k:k+1])['WAPE']:.2f}" for k in range(4)) for p in PRED.values()]
print(tabla.sort_values("MASE"))
ax = tabla["MASE"].sort_values().plot.barh(title="MASE (menor = mejor; 1 = ingenuo estacional)", figsize=(7, 2.8))
ax.axvline(1, color="k", ls="--"); plt.show()

### ¿Por qué gana (o no) el global?
Mira el reparto del WAPE por categoría y tamaño de serie. Normalmente el global aprovecha dos cosas que el local no tiene: **información compartida** entre series parecidas (sobre todo en las pequeñas, muy ruidosas) y **variables externas** (promos y festivos).

In [ ]:
# ¿Dónde gana el modelo global? Por categoría y por tamaño de serie
cat = SERIES["categoria"].values
nivel = Y[:, :ORIGENES[-1] + 1][:, -28:].mean(1)
tam = pd.qcut(nivel, 3, labels=["pequeña", "mediana", "grande"])
filas_t = []
for grupo, mask in [(f"cat {c}", cat == c) for c in PRODUCTOS] + [(f"serie {t}", np.asarray(tam) == t) for t in ["pequeña", "mediana", "grande"]]:
    fila = {"grupo": grupo}
    for n in ("estacional", "ETS local", "global HGB"):
        fila[n] = metricas(REAL[:, mask], PRED[n][:, mask], ESC[:, mask])["WAPE"]
    filas_t.append(fila)
print(pd.DataFrame(filas_t).set_index("grupo").round(3))

## 5 · Intervalos de predicción

Un número sin incertidumbre no sirve para decidir stock. Comparamos dos enfoques para un intervalo del 80 %:

- **Regresión cuantílica:** entrenar modelos que predicen directamente los cuantiles 10 % y 90 % (`loss="quantile"`). Flexible, pero cuesta un modelo por cuantil y puede quedar mal calibrado.
- **Conformal (split):** tomar un modelo cualquiera y calibrar el intervalo con los **errores reales fuera de muestra** de un periodo anterior. Aquí usamos el pliegue previo del backtesting, escalando el error y agrupándolo por tamaño de serie (la incertidumbre relativa de una serie pequeña es mucho mayor). Su garantía de cobertura vale si el futuro se parece a la calibración; si cambia el régimen, se pierde.

Se evalúan con la **cobertura** (¿cae el real dentro el 80 % de las veces?), el **ancho** (intervalos más estrechos = más útiles) y el *interval score* (penaliza anchura y fallos en una sola cifra).

In [ ]:
K = 3                                                    # intervalos en el último pliegue (0-based = 3)
o = ORIGENES[K]
# 1) Regresión cuantílica: dos modelos más (q10 y q90) -> intervalo central del 80 %
t0 = time.time()
Xo = filas([o], con_objetivo=False)
cuant = {}
for q in (0.1, 0.9):
    m, _ = ajustar_global(o, perdida="quantile", quantile=q, max_iter=150)
    cuant[q] = np.clip(m.predict(Xo[VARS]) * Xo["esc"].values, 0, None).reshape(H, S).T
print(f"cuantiles listos en {time.time() - t0:.0f} s")

# 2) Conformal: residuos del pliegue ANTERIOR (fuera de muestra), escalados y agrupados por tamaño
def escala_nivel(k):
    return Y[:, ORIGENES[k] - 27:ORIGENES[k] + 1].mean(1) + 0.5

def conformal(k, alfa=0.2, base="global HGB"):
    e_cal = (REAL[k - 1] - PRED[base][k - 1]) / escala_nivel(k - 1)[:, None]
    cortes = np.quantile(escala_nivel(k - 1), [1 / 3, 2 / 3])
    g_cal, g = np.digitize(escala_nivel(k - 1), cortes), np.digitize(escala_nivel(k), cortes)
    lo, hi = np.zeros(S), np.zeros(S)
    for grupo in range(3):
        r = e_cal[g_cal == grupo].ravel()
        lo[g == grupo], hi[g == grupo] = np.quantile(r, [alfa / 2, 1 - alfa / 2])
    p = PRED[base][k]
    return np.clip(p + lo[:, None] * escala_nivel(k)[:, None], 0, None), p + hi[:, None] * escala_nivel(k)[:, None]

def evalua_intervalo(lo, hi, real, alfa=0.2):
    cobertura = float(((real >= lo) & (real <= hi)).mean())
    ancho = float((hi - lo).sum() / real.sum())
    puntuacion = (hi - lo) + (2 / alfa) * np.clip(lo - real, 0, None) + (2 / alfa) * np.clip(real - hi, 0, None)
    return {"cobertura": round(cobertura, 3), "ancho/ventas": round(ancho, 3),
            "interval score/ventas": round(float(puntuacion.sum() / real.sum()), 3)}

resumen = {"cuantílica q10-q90": evalua_intervalo(cuant[0.1], cuant[0.9], REAL[K])}
lo_c, hi_c = conformal(K)
resumen["conformal (pliegue previo)"] = evalua_intervalo(lo_c, hi_c, REAL[K])
print("Objetivo de cobertura: 0.80")
print(pd.DataFrame(resumen).T)
for k in (1, 2):                                                    # estabilidad: el conformal en otros pliegues
    lo_k, hi_k = conformal(k)
    print(f"conformal en el pliegue {k + 1}:", evalua_intervalo(lo_k, hi_k, REAL[k]))

i = int(np.argmax(nivel))                                          # una serie grande, para verla
dias = FECHAS[o + 1:o + 1 + H]
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(FECHAS[o - 41:o + 1], Y[i, o - 41:o + 1], "k", label="histórico")
ax.plot(dias, REAL[K][i], "k--", label="real")
ax.plot(dias, PRED["global HGB"][K][i], label="global HGB")
ax.fill_between(dias, lo_c[i], hi_c[i], alpha=0.25, label="conformal 80 %")
ax.set_title(f"{SERIES.producto[i]} · {SERIES.tienda[i]}"); ax.legend(ncol=4); plt.show()

## 6 · Jerarquías y reconciliación

Si pronosticas cada nivel por separado, las cifras **no cuadran**: la suma de las tiendas no es el total. Reconciliar = ajustar las previsiones para que sean coherentes.

Con la matriz de suma **S** (cada fila agregada = suma de series base), todo conjunto coherente es `y = S · b`, siendo `b` las series base. Opciones:
- **Bottom-up:** sumar las series base. Simple y sin sesgo de agregación, pero depende de lo ruidosas que sean las series base.
- **Reconciliación por mínimos cuadrados:** partir de previsiones de **todos** los niveles y proyectarlas sobre el espacio coherente. OLS trata todas por igual; WLS pondera (aquí con pesos estructurales: más peso a las series más agregadas). La variante más usada en la literatura es MinT, que usa la covarianza de los errores (ver recursos).

Probamos dos escenarios: previsiones base **solo ETS** en los 110 niveles, y una mezcla (modelo global fuerte abajo, ETS arriba).

In [ ]:
# Matriz de suma S: cada fila agregada es la suma de series de base (producto x tienda)
nombres = ["Total"] + [f"tienda:{t}" for t in TIENDAS] + [f"categoría:{c}" for c in PRODUCTOS] + ["base"] * S
nivel_de = ["total"] + ["tienda"] * len(TIENDAS) + ["categoría"] * len(PRODUCTOS) + ["base"] * S
S_mat = np.vstack([
    np.ones((1, S)),
    np.array([(SERIES["tienda"] == t).values for t in TIENDAS], float),
    np.array([(SERIES["categoria"] == c).values for c in PRODUCTOS], float),
    np.eye(S),
])
nivel_de = np.array(nivel_de)
print("S:", S_mat.shape, "-> 1 total + 5 tiendas + 4 categorías + 100 series base = 110 series")

def reconcilia(base_fc, W=None):
    """Reconciliación por mínimos cuadrados (OLS si W=None; WLS con pesos W): y_rec = S (S'W⁻¹S)⁻¹ S'W⁻¹ y_base."""
    Wi = np.eye(len(S_mat)) if W is None else np.diag(1 / W)
    G = np.linalg.solve(S_mat.T @ Wi @ S_mat, S_mat.T @ Wi)
    return S_mat @ G @ base_fc

def wape_por_nivel(fc, real_110):
    return {n: float(np.abs(fc[nivel_de == n] - real_110[nivel_de == n]).sum() / real_110[nivel_de == n].sum())
            for n in ["total", "tienda", "categoría", "base"]}

METODOS = ["ETS · independiente", "ETS · bottom-up", "ETS · OLS", "ETS · WLS",
           "HGB base + ETS arriba · independiente", "HGB base + ETS arriba · WLS", "HGB · bottom-up"]
resultados = {m: [] for m in METODOS}
incoherencia = []
for k, o in enumerate(ORIGENES):
    real_110 = S_mat @ REAL[k]
    agregadas_ets = pred_ets(S_mat[:10] @ Y[:, :o + 1])              # ETS sobre las 10 series agregadas
    ets = np.vstack([agregadas_ets, PRED["ETS local"][k]])           # previsiones base solo con ETS
    mixto = np.vstack([agregadas_ets, PRED["global HGB"][k]])        # base fuerte abajo, ETS arriba
    resultados["ETS · independiente"].append(wape_por_nivel(ets, real_110))
    resultados["ETS · bottom-up"].append(wape_por_nivel(S_mat @ PRED["ETS local"][k], real_110))
    resultados["ETS · OLS"].append(wape_por_nivel(reconcilia(ets), real_110))
    resultados["ETS · WLS"].append(wape_por_nivel(reconcilia(ets, W=S_mat.sum(1)), real_110))
    resultados["HGB base + ETS arriba · independiente"].append(wape_por_nivel(mixto, real_110))
    resultados["HGB base + ETS arriba · WLS"].append(wape_por_nivel(reconcilia(mixto, W=S_mat.sum(1)), real_110))
    resultados["HGB · bottom-up"].append(wape_por_nivel(S_mat @ PRED["global HGB"][k], real_110))
    incoherencia.append(float(np.abs(ets[0] - ets[1:6].sum(0)).sum() / ets[0].sum()))

tab = pd.DataFrame({m: pd.DataFrame(v).mean() for m, v in resultados.items()}).T.round(3)
print("WAPE medio de los 4 pliegues, por nivel de la jerarquía")
print(tab)
print(f"\nIncoherencia de las previsiones base ETS (|total - suma de tiendas| / total): {np.mean(incoherencia):.1%}")


**Lectura honesta del resultado:** reconciliar da *coherencia* (las cifras cuadran, y eso ya es un requisito de negocio) y suele mejorar un poco las previsiones débiles, pero **no hace magia**: si tu modelo de base es mucho mejor que el de los niveles altos, sumarlo (bottom-up) puede ganar a todo lo demás. Se decide con backtesting, nivel por nivel.

## 7 · Modelos fundacionales de series temporales (conceptual + esqueleto Colab)

Son modelos preentrenados con enormes colecciones de series que pronostican series **que nunca han visto**, sin entrenar (*zero-shot*) o con un ajuste ligero. Ejemplos de familias: Chronos (Amazon), TimesFM (Google), Moirai (Salesforce), Lag-Llama o TimeGPT (Nixtla, de pago por API). **Comprueba cuál es el modelo y la versión vigentes** antes de usarlos: el campo cambia cada pocos meses.

**Cuándo tienen sentido:** series cortas o nuevas (producto recién lanzado), muchas series sin tiempo de modelar cada una, un *baseline* fuerte en minutos. **Cuándo no ganan:** cuando tu señal está en variables externas (promos, precio, festivos locales: muchos fundacionales no las usan o las usan poco) y tienes mucho histórico propio.

**La regla de oro es la de siempre:** se evalúan **con el mismo backtesting, las mismas métricas y frente al ingenuo estacional y a tu modelo global**. Abajo, la función `evaluar_modelo` que ya sirve para enchufar cualquier modelo (aquí con una referencia, para que se ejecute sin descargas).

In [ ]:
def evaluar_modelo(nombre, f_pred):
    """Enchufa cualquier modelo: f_pred(historia[series, tiempo]) -> previsión [series, H].
    Mismos pliegues y mismas métricas que el resto, sin mirar el futuro."""
    p = np.stack([f_pred(Y[:, :o + 1]) for o in ORIGENES])
    PRED[nombre] = p
    return {nombre: metricas(REAL, p, ESC)}

print(evaluar_modelo("(demo) estacional 4s", pred_estacional_4s))

# ---- ESQUELETO PARA COLAB (no se ejecuta aquí: requiere descargar pesos) -------------------------
# Comprueba antes el paquete y el nombre del modelo vigentes en la documentación del autor.
#
# !pip install chronos-forecasting
# import torch
# from chronos import BaseChronosPipeline
# pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-small", device_map="cpu")
#
# def pred_chronos(historia):
#     contexto = [torch.tensor(s[-364:], dtype=torch.float32) for s in historia]   # una serie por elemento
#     _, media = pipeline.predict_quantiles(contexto, prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])
#     return np.clip(media.numpy(), 0, None)                                       # (series, H)
#
# print(evaluar_modelo("Chronos (zero-shot)", pred_chronos))
# Compáralo con "estacional", "ETS local" y "global HGB". ¿Mejora donde hay promos? ¿Y en series pequeñas?

## Ejercicios

Escribe tu código en las celdas vacías. Usa lo definido arriba (`PRED`, `REAL`, `ESC`, `metricas`, `ajustar_global`, `conformal`, `S_mat`, `reconcilia`…).

1. **MASE a mano.** Para la serie 0 (`SERIES.iloc[0]`) y el último pliegue, calcula el MASE del ingenuo estacional y del global sin usar `metricas`. ¿Qué significa que uno sea > 1? Calcula también el WAPE del global por **tienda**.
2. **Función de pérdida.** Reentrena el global del último pliegue con `perdida="squared_error"` y compara MASE, WAPE y sesgo con el de Poisson. ¿Por qué podría convenir Poisson con ventas pequeñas?
3. **Cobertura por semana de horizonte.** Con los intervalos conformal y cuantílico del último pliegue, calcula la cobertura en las semanas 1–4. ¿Se degrada con el horizonte?
4. **Top-down.** Implementa la reconciliación *top-down por proporciones*: pronostica solo el total (ETS) y reparte según la proporción de cada serie base en las los últimos 28 días. Compara su WAPE a nivel base con el del global. ¿Qué pierde?
5. **Ensamble.** Combina `w · ETS + (1 − w) · global`. Elige `w` con los pliegues 1–2 y evalúa en los pliegues 3–4. ¿Por qué no se debe elegir `w` con los mismos pliegues en que se evalúa?

In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí
